# CineEmbed — 07 Round 1: architecture comparison @ z=64

Round 1 of the locked two-round modeling strategy (spec `docs/superpowers/specs/2026-05-16-two-round-modeling-strategy.md`). Runs **three new architecture-family configs** at z=64 to compare against the MVP carry-over runs already on Drive. Selection metric: `geo_NMI = (gNMI · dNMI · lNMI)^(1/3)`.

**Configs (~30 min total on T4):**

| run_name | family | source | notes |
|---|---|---|---|
| `vae_z64` | VAE | cold start, β warmup 10 epochs | methodological completeness |
| `dec_z64_k21_from_contrastive_t0p1` | AE→DEC fine-tune | `contrastive_tau0p1_drop0p3/pretext_backbone.pt` | hero candidate A |
| `dec_z64_k21_from_contrastive_t0p5` | AE→DEC fine-tune | `contrastive_tau0p5_drop0p3/pretext_backbone.pt` | hero candidate B |

**Outputs per run (under `<artifacts>/models/<run_name>/`):**
- DEC family: `ae.pt`, `dec.pt`, `backbone.pt` (encoder-only, for inference), `eval.json`
- VAE family: `vae.pt`, `backbone.pt`, `eval.json`

All metrics + artifacts go to a single W&B run per config under `group=round-1`.

**Prerequisites:** Phase 1 contrastive sweep complete (`03_train_contrastive.ipynb`) — both `contrastive_tau0p1_drop0p3` and `contrastive_tau0p5_drop0p3` checkpoints must exist on Drive.

Run `00_colab_setup.ipynb` once per session first.

In [1]:
import os, sys, json, time
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    try:
        from google.colab import userdata
        _token = userdata.get('GITHUB_TOKEN')
        REPO_URL = f"https://{_token}@github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"
    except Exception:
        REPO_URL = "https://github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"
    REPO_ROOT = Path('/content/cineembed-repo')
    ARTIFACTS = Path('/content/drive/MyDrive/CineEmbed/artifacts')
    if not REPO_ROOT.exists():
        get_ipython().system(f'git clone {REPO_URL} {REPO_ROOT}')
    else:
        get_ipython().system(f'cd {REPO_ROOT} && git fetch -q && git checkout main -q && git pull -q')
    get_ipython().system(f'pip install -e {REPO_ROOT} -q')
    get_ipython().system('pip install wandb -q')
else:
    REPO_ROOT = Path('..').resolve()
    ARTIFACTS = REPO_ROOT / 'artifacts'

sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import torch
import torch.nn as nn

from cineembed import data, backbone, heads, losses, train, eval as cev

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Repo:      {REPO_ROOT}")
print(f"Artifacts: {ARTIFACTS}")
print(f"Device:    {DEVICE}")

# Sanity-check artifacts + Phase 1 prerequisites
for p in [ARTIFACTS / 'feature_matrix.npz', ARTIFACTS / 'movies_eda_final.csv']:
    assert p.exists(), f"Missing artifact: {p}"
for prereq in ['contrastive_tau0p1_drop0p3', 'contrastive_tau0p5_drop0p3']:
    p = ARTIFACTS / 'models' / prereq / 'pretext_backbone.pt'
    assert p.exists(), f"Missing Phase 1 prerequisite: {p} — run 03_train_contrastive.ipynb first"

get_ipython().system(f'cd {REPO_ROOT} && git log --oneline -1')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cineembed (pyproject.toml) ... done
Repo:      /content/cineembed-repo
Artifacts: /content/drive/MyDrive/CineEmbed/artifacts
Device:    cuda
276f47d (HEAD -> main, origin/main) feat(round-1): notebook for architecture comparison @ z=64


## W&B configuration (auto-detected — no interactive prompt)

Detection order: Colab Secret `WANDB_API_KEY` → `os.environ['WANDB_API_KEY']` → `~/.netrc`. Falls back to offline mode if all three fail; never prompts stdin.

In [2]:
WANDB_PROJECT = 'cineembed'       # set None to skip W&B entirely
WANDB_ENTITY  = None              # e.g. 'team3-seng474' — None = default from netrc
WANDB_GROUP   = 'round-1'         # group all 3 runs together on the dashboard

WANDB_MODE = None
WANDB_KEY_PRESENT = False

if WANDB_PROJECT and IN_COLAB:
    try:
        from google.colab import userdata
        _key = userdata.get('WANDB_API_KEY')
        if _key:
            os.environ['WANDB_API_KEY'] = _key
            WANDB_KEY_PRESENT = True
    except Exception:
        pass

if WANDB_PROJECT and not WANDB_KEY_PRESENT and os.environ.get('WANDB_API_KEY'):
    WANDB_KEY_PRESENT = True

if WANDB_PROJECT and not WANDB_KEY_PRESENT and not IN_COLAB:
    if Path.home().joinpath('.netrc').exists():
        WANDB_KEY_PRESENT = True

if WANDB_PROJECT is None:
    WANDB_MODE = 'disabled'
    print('W&B disabled (WANDB_PROJECT is None).')
elif WANDB_KEY_PRESENT:
    WANDB_MODE = 'online'
    os.environ['WANDB_MODE'] = 'online'
    print(f"W&B ONLINE  project='{WANDB_PROJECT}'  group='{WANDB_GROUP}'")
else:
    WANDB_MODE = 'offline'
    os.environ['WANDB_MODE'] = 'offline'
    print('No WANDB_API_KEY found → using OFFLINE mode.')
    print(f"Offline runs land under {REPO_ROOT}/wandb/; sync later with `wandb sync`.")

print(f"\nFinal: WANDB_MODE={WANDB_MODE}")

No WANDB_API_KEY found → using OFFLINE mode.
Offline runs land under /content/cineembed-repo/wandb/; sync later with `wandb sync`.

Final: WANDB_MODE=offline


## Data + globals
Same data layout as 02_train_ae / 04_train_dec — single load, reused across all 3 configs.

In [3]:
X, feature_names = data.load_feature_matrix(ARTIFACTS / 'feature_matrix.npz')
labels = data.get_labels(ARTIFACTS / 'movies_eda_final.csv')
block_slices = data.get_block_indices(feature_names)
has_bio = X[:, block_slices['director'].start + 64].clone()

BLOCK_DIMS = {b: (slc.stop - slc.start) for b, slc in block_slices.items()}
PROJ_DIMS = backbone.DEFAULT_PROJ_DIMS
w_blocks = losses.compute_block_weights(X.numpy(), block_slices, w_min=0.1, w_max=10.0)

train_idx, val_idx = data.train_val_split(X.shape[0], val_frac=0.1, seed=42)
train_loader = data.make_dataloader(X, has_bio, batch_size=512, shuffle=True,
                                     indices=train_idx, block_slices=block_slices, seed=42)
val_loader = data.make_dataloader(X, has_bio, batch_size=512, shuffle=False,
                                    indices=val_idx, block_slices=block_slices, seed=42)

print(f"X: {X.shape}  has_bio_sum: {int(has_bio.sum())}")
print(f"BLOCK_DIMS: {BLOCK_DIMS}")
print(f"w_blocks (W2): {w_blocks}")

X: torch.Size([329044, 564])  has_bio_sum: 10458
BLOCK_DIMS: {'numerical': 6, 'genre': 22, 'language': 31, 'decade': 2, 'awards': 6, 'text': 384, 'director': 113}
w_blocks (W2): {'numerical': 0.16672051367502017, 'genre': 0.7100368298560866, 'language': 1.7986772489608718, 'decade': 0.5000579722378256, 'awards': 0.16650737001734822, 'text': 1.351932282578171, 'director': 1.0098946369067103}


## Helpers
Three training primitives (AE fine-tune from optional pretext backbone, DEC fine-tune from trained AE, VAE cold-start with β warmup) and a shared eval helper. All match the patterns in `02_train_ae.ipynb` and `04_train_dec.ipynb`.

In [4]:
# ── Optional W&B per-epoch logger (no-op if wandb_run is None) ────────────────
from cineembed.wandb_integration import log_epoch as _log_epoch
from cineembed.wandb_integration import log_eval as _log_eval
from cineembed.wandb_integration import log_artifact as _log_artifact
from cineembed.wandb_integration import wandb_run as _wandb_run
from contextlib import nullcontext


def _build_backbone(latent_dim: int = 64, hidden_dim: int = 128, dropout: float = 0.2):
    return backbone.MultiModalBackbone(
        BLOCK_DIMS,
        proj_dims=PROJ_DIMS,
        hidden_dim=hidden_dim,
        latent_dim=latent_dim,
        dropout=dropout,
    )


def train_ae_finetune(run_name, pretext_backbone_path=None, *,
                       n_epochs=100, wandb_run=None):
    """Build a fresh AEHead, optionally load pretext backbone weights, train end-to-end.
    Decoder is always randomly initialized. Returns (ae_head, history).
    """
    torch.manual_seed(42)
    bb = _build_backbone()
    if pretext_backbone_path is not None:
        sd = torch.load(pretext_backbone_path, map_location='cpu', weights_only=True)
        missing, unexpected = bb.load_state_dict(sd, strict=False)
        if missing or unexpected:
            print(f"  [warn] state_dict load: missing={len(missing)} unexpected={len(unexpected)}")
        print(f"  [init] loaded pretext backbone from {Path(pretext_backbone_path).parent.name}/")
    ae_head = heads.AEHead(bb, BLOCK_DIMS, PROJ_DIMS, hidden_dim=128)

    def loss_fn(model, batch):
        decoded = model(batch['blocks'])
        return losses.weighted_recon_loss(decoded, batch['blocks'],
                                            batch['has_bio'], w_blocks)

    ckpt = ARTIFACTS / 'models' / run_name / 'ae.pt'
    ckpt.parent.mkdir(parents=True, exist_ok=True)

    history = train.train_model(
        model=ae_head, loss_fn=loss_fn,
        train_loader=train_loader, val_loader=val_loader,
        n_epochs=n_epochs, lr=1e-3, weight_decay=1e-5,
        early_stop_patience=10, early_stop_min_delta=1e-4,
        gradient_clip_norm=1.0,
        checkpoint_path=ckpt, device=DEVICE, seed=42,
        wandb_run=wandb_run,
    )
    train.load_checkpoint(ae_head, ckpt, device=DEVICE)
    return ae_head, history


def _compute_full_latents_and_counts(dec_model, device):
    """Forward pass over the entire dataset; return (z_array, per-cluster counts).
    Mirror of 04_train_dec.ipynb helper.
    """
    dec_model.eval()
    full_loader = data.make_dataloader(X, has_bio, batch_size=2048, shuffle=False,
                                         block_slices=block_slices)
    z_chunks, q_argmax_chunks = [], []
    with torch.no_grad():
        for batch in full_loader:
            blk = {k_: v.to(device) for k_, v in batch['blocks'].items()}
            z, _, q = dec_model(blk)
            z_chunks.append(z.cpu().numpy())
            q_argmax_chunks.append(q.argmax(dim=1).cpu().numpy())
    z_all = np.concatenate(z_chunks, axis=0)
    assignments = np.concatenate(q_argmax_chunks, axis=0)
    counts = np.bincount(assignments, minlength=dec_model.n_clusters)
    return z_all, torch.from_numpy(counts)


def train_dec_finetune(run_name, ae_head, *, k=21, n_epochs=50,
                        cluster_size_floor=0.001, wandb_run=None):
    """Build DECHead from a trained AEHead, train with per-epoch reinit (spec §10).
    Mirror of 04_train_dec.ipynb train_dec_run.
    """
    torch.manual_seed(42)
    z_dim = ae_head.backbone.latent_dim

    # ── Init centers via KMeans on AE latents over the full dataset ──────────
    ae_head.eval()
    z_init_chunks = []
    full_loader = data.make_dataloader(X, has_bio, batch_size=2048, shuffle=False,
                                         block_slices=block_slices)
    with torch.no_grad():
        for batch in full_loader:
            blk = {k_: v.to(DEVICE) for k_, v in batch['blocks'].items()}
            z_init_chunks.append(ae_head.encode(blk).cpu().numpy())
    z_init = np.concatenate(z_init_chunks, axis=0)

    dec_head = heads.DECHead(ae_head.backbone, ae_head.decoder,
                              n_clusters=k, latent_dim=z_dim)
    dec_head.initialize_centers(z_init, seed=42)
    dec_head = dec_head.to(DEVICE)

    opt = torch.optim.Adam(dec_head.parameters(), lr=1e-3, weight_decay=1e-5)
    history = {'train_loss': [], 'val_loss': [], 'n_reinit': []}
    best_val = float('inf'); patience = 0
    ckpt_path = ARTIFACTS / 'models' / run_name / 'dec.pt'
    n_total = X.shape[0]

    for epoch in range(n_epochs):
        _t_epoch = time.time()
        # ── train ──
        dec_head.train()
        train_losses = []
        for batch in train_loader:
            blk = {k_: v.to(DEVICE) for k_, v in batch['blocks'].items()}
            hb = batch['has_bio'].to(DEVICE)
            opt.zero_grad()
            z, decoded, _ = dec_head(blk)
            loss, _, _ = losses.dec_loss(z, decoded, blk, dec_head.cluster_centers,
                                           hb, w_blocks, lambda_recon=0.1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(dec_head.parameters(), 1.0)
            opt.step()
            train_losses.append(float(loss.item()))
        train_avg = sum(train_losses) / max(len(train_losses), 1)
        history['train_loss'].append(train_avg)

        # ── cluster-collapse reinit (spec §10) ──
        z_full, cluster_counts = _compute_full_latents_and_counts(dec_head, DEVICE)
        n_reinit = dec_head.reinit_collapsed_centers(
            cluster_counts=cluster_counts, z_pool=z_full,
            n_total=n_total, size_floor=cluster_size_floor, seed=42 + epoch,
        )
        history['n_reinit'].append(n_reinit)

        # ── val ──
        dec_head.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                blk = {k_: v.to(DEVICE) for k_, v in batch['blocks'].items()}
                hb = batch['has_bio'].to(DEVICE)
                z, decoded, _ = dec_head(blk)
                v_loss, _, _ = losses.dec_loss(z, decoded, blk, dec_head.cluster_centers,
                                                 hb, w_blocks, lambda_recon=0.1)
                val_losses.append(float(v_loss.item()))
        val_avg = sum(val_losses) / max(len(val_losses), 1)
        history['val_loss'].append(val_avg)

        elapsed = time.time() - _t_epoch
        improved = (best_val - val_avg) > 1e-4
        marker = '↓' if improved else '·'
        print(f"  [DEC] epoch {epoch+1:3d}/{n_epochs} | train={train_avg:.4f} val={val_avg:.4f} {marker} "
              f"reinit={n_reinit} | {elapsed:.1f}s")

        # ── W&B log (offset epoch by a large constant so AE + DEC don't collide on x-axis) ──
        _log_epoch(wandb_run, epoch=10_000 + epoch, train_loss=train_avg, val_loss=val_avg,
                    lr=1e-3, extra={'dec_n_reinit': n_reinit, 'stage': 1})

        if improved:
            best_val = val_avg; patience = 0
            ckpt_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save({'model_state': dec_head.state_dict(), 'epoch': epoch,
                        'val_loss': val_avg, 'history': history}, ckpt_path)
        else:
            patience += 1
            if patience >= 8:
                print(f"  [DEC] early stop at epoch {epoch+1}")
                break

    train.load_checkpoint(dec_head, ckpt_path, device=DEVICE)
    history['final_val_loss'] = best_val
    history['total_reinit'] = int(sum(history['n_reinit']))
    return dec_head, history


def train_vae(run_name, *, n_epochs=100, beta_max=0.1, beta_warmup=10,
               wandb_run=None):
    """VAE z=64 cold start with β warmup. Returns (vae_head, history)."""
    torch.manual_seed(42)
    bb = _build_backbone()
    vae_head = heads.VAEHead(bb, BLOCK_DIMS, PROJ_DIMS, hidden_dim=128)

    def loss_fn(model, batch, epoch):
        decoded, mu, log_var = model(batch['blocks'])
        beta = min(beta_max, beta_max * (epoch + 1) / max(beta_warmup, 1))
        loss, _, _ = losses.vae_elbo(decoded, batch['blocks'], mu, log_var,
                                       batch['has_bio'], w_blocks, beta=beta)
        return loss

    ckpt = ARTIFACTS / 'models' / run_name / 'vae.pt'
    ckpt.parent.mkdir(parents=True, exist_ok=True)

    history = train.train_model(
        model=vae_head, loss_fn=loss_fn,
        train_loader=train_loader, val_loader=val_loader,
        n_epochs=n_epochs, lr=1e-3, weight_decay=1e-5,
        early_stop_patience=10, early_stop_min_delta=1e-4,
        gradient_clip_norm=1.0,
        checkpoint_path=ckpt, device=DEVICE, seed=42,
        wandb_run=wandb_run,
    )
    train.load_checkpoint(vae_head, ckpt, device=DEVICE)
    return vae_head, history


@torch.no_grad()
def encode_all_latents(any_head, device):
    """Extract deterministic latent z from any head. VAEHead.encode returns (mu, log_var) — use mu."""
    any_head.eval()
    full_loader = data.make_dataloader(X, has_bio, batch_size=2048, shuffle=False,
                                         block_slices=block_slices)
    chunks = []
    for batch in full_loader:
        blk = {k_: v.to(device) for k_, v in batch['blocks'].items()}
        if isinstance(any_head, heads.VAEHead):
            mu, _ = any_head.encode(blk)
            z_out = mu
        else:
            z_out = any_head.encode(blk)
        chunks.append(z_out.cpu().numpy())
    return np.concatenate(chunks, axis=0)


def do_eval(z, dec_model=None, *, seed=42):
    """KMeans + GMM + per-axis-k + multilabel macro-NMI. Optionally DEC argmax.
    Matches the eval shape produced by scripts/train_contrastive.py for direct comparison.
    """
    kmeans_assign = cev.cluster_assignments_kmeans(z, k=21, seed=seed)
    gmm_assign = cev.cluster_assignments_gmm(z, k=21, seed=seed)
    out = {
        'kmeans_k21':        cev.evaluate_run(kmeans_assign, labels),
        'gmm_k21':           cev.evaluate_run(gmm_assign, labels),
        'per_axis_k_kmeans': cev.evaluate_run_per_axis_k(z, labels, seed=seed),
    }
    g = block_slices['genre']
    genre_onehot = X[:, g.start:g.start + 21].numpy().astype(np.float32)
    out['multilabel_genre_macro_nmi_kmeans'] = cev.multilabel_macro_nmi(
        kmeans_assign, genre_onehot, metric='nmi')

    if dec_model is not None:
        full_batches = list(data.make_dataloader(X, has_bio, batch_size=2048, shuffle=False,
                                                   block_slices=block_slices))
        dec_assign = cev.cluster_assignments_dec(dec_model, full_batches, device=DEVICE)
        out['dec_argmax'] = cev.evaluate_run(dec_assign, labels)
    return out


def print_eval_summary(name, eval_out):
    km = eval_out['kmeans_k21']
    print('=' * 64)
    print(f"[{name}] KMeans  k=21    g={km['genre_nmi']:.3f}  d={km['decade_nmi']:.3f}  l={km['lang_nmi']:.3f}")
    gm = eval_out['gmm_k21']
    print(f"[{name}] GMM     k=21    g={gm['genre_nmi']:.3f}  d={gm['decade_nmi']:.3f}  l={gm['lang_nmi']:.3f}")
    if 'dec_argmax' in eval_out:
        d = eval_out['dec_argmax']
        print(f"[{name}] DEC argmax     g={d['genre_nmi']:.3f}  d={d['decade_nmi']:.3f}  l={d['lang_nmi']:.3f}")
    print(f"[{name}] MVP baseline   g=0.332 (dec_z64_k21)")
    print('=' * 64)


def push_eval_to_wandb(run, eval_out, run_dir, run_name, family):
    """Push all eval metrics + headline summary keys + artifact to a single wandb run."""
    if run is None:
        return
    _log_eval(run, eval_out['kmeans_k21'],         prefix='km_')
    _log_eval(run, eval_out['gmm_k21'],            prefix='gmm_', add_geo_nmi=False)
    pa = eval_out.get('per_axis_k_kmeans', {})
    if pa:
        _log_eval(run, pa, prefix='axis_', add_geo_nmi=False)
    ml = eval_out.get('multilabel_genre_macro_nmi_kmeans', {})
    if isinstance(ml, dict) and 'macro_nmi' in ml:
        run.log({'km_multilabel_macro_nmi': float(ml['macro_nmi'])})
    if 'dec_argmax' in eval_out:
        _log_eval(run, eval_out['dec_argmax'], prefix='dec_', add_geo_nmi=True)

    # Run-summary headline numbers (surface on the dashboard run list)
    km = eval_out['kmeans_k21']
    run.summary['headline_km_genre_nmi'] = float(km['genre_nmi'])
    if all(kk in km for kk in ('genre_nmi','decade_nmi','lang_nmi')):
        prod = max(km['genre_nmi']*km['decade_nmi']*km['lang_nmi'], 0.0)
        run.summary['headline_km_geo_nmi'] = prod**(1/3) if prod > 0 else 0.0
    if 'dec_argmax' in eval_out:
        d = eval_out['dec_argmax']
        run.summary['headline_dec_genre_nmi'] = float(d['genre_nmi'])
        if all(kk in d for kk in ('genre_nmi','decade_nmi','lang_nmi')):
            prod = max(d['genre_nmi']*d['decade_nmi']*d['lang_nmi'], 0.0)
            run.summary['headline_dec_geo_nmi'] = prod**(1/3) if prod > 0 else 0.0

    # Artifact — final model state + backbone-only encoder for downstream inference
    if family == 'dec_from_contrastive':
        for fn in ('ae.pt', 'dec.pt', 'backbone.pt'):
            p = run_dir / fn
            if p.exists():
                _log_artifact(run, p, name=f'{fn[:-3]}__{run_name}', type='model')
    elif family == 'vae':
        for fn in ('vae.pt', 'backbone.pt'):
            p = run_dir / fn
            if p.exists():
                _log_artifact(run, p, name=f'{fn[:-3]}__{run_name}', type='model')


print('Helpers defined: train_ae_finetune, train_dec_finetune, train_vae, encode_all_latents, do_eval, push_eval_to_wandb')

Helpers defined: train_ae_finetune, train_dec_finetune, train_vae, encode_all_latents, do_eval, push_eval_to_wandb


## Configs + sweep
Per config: open one wandb run (or null-context when disabled), train, save backbone-only for inference, eval, log everything. Idempotent — set `FORCE=True` to retrain.

Expected wall-clock: ~10-15 min per config on T4 (`vae_z64` ~10 min; each AE→DEC ~12 min split between stages). Total ~35-45 min.

In [5]:
ROUND1_CONFIGS = [
    {
        'run_name': 'vae_z64',
        'family':   'vae',
        'pretext_source': None,
        'k': None,
    },
    {
        'run_name': 'dec_z64_k21_from_contrastive_t0p1',
        'family':   'dec_from_contrastive',
        'pretext_source': ARTIFACTS / 'models' / 'contrastive_tau0p1_drop0p3' / 'pretext_backbone.pt',
        'k': 21,
    },
    {
        'run_name': 'dec_z64_k21_from_contrastive_t0p5',
        'family':   'dec_from_contrastive',
        'pretext_source': ARTIFACTS / 'models' / 'contrastive_tau0p5_drop0p3' / 'pretext_backbone.pt',
        'k': 21,
    },
]

FORCE = False  # True → retrain even if eval.json exists

all_results = {}
for cfg in ROUND1_CONFIGS:
    run_name = cfg['run_name']
    family   = cfg['family']
    run_dir  = ARTIFACTS / 'models' / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    evalj    = run_dir / 'eval.json'

    print('\n' + '#' * 78)
    print(f"# {run_name}  (family={family}, mode={WANDB_MODE})")
    print('#' * 78)

    if (not FORCE) and evalj.exists():
        print(f"SKIP — already done. eval={evalj.stat().st_size} B.")
        print(f"      Set FORCE=True to overwrite, or delete {run_dir} to redo.")
        all_results[run_name] = json.loads(evalj.read_text())
        continue

    _t_run = time.time()

    # W&B context (single run for the whole train+eval cycle)
    wandb_cfg = {
        'phase':           'round-1',
        'family':          family,
        'pretext_source':  str(cfg['pretext_source']).split('/')[-2] if cfg.get('pretext_source') else None,
        'k':               cfg.get('k'),
        'latent_dim':      64,
        'hidden_dim':      128,
        'batch_size':      512,
    }
    if WANDB_PROJECT:
        ctx = _wandb_run(
            config=wandb_cfg, run_name=run_name,
            project=WANDB_PROJECT, entity=WANDB_ENTITY, group=WANDB_GROUP,
            tags=['round-1', family], mode=WANDB_MODE,
            notes=f'Round 1 ({family}). Spec 2026-05-16-two-round-modeling-strategy.md.',
        )
    else:
        ctx = nullcontext(None)

    with ctx as run:
        if family == 'vae':
            vae_head, hist = train_vae(run_name, wandb_run=run)
            z = encode_all_latents(vae_head, DEVICE)
            # Save backbone-only state for downstream inference convenience
            torch.save(vae_head.backbone.state_dict(), run_dir / 'backbone.pt')
            eval_out = do_eval(z, dec_model=None)

        elif family == 'dec_from_contrastive':
            ae_head, ae_hist = train_ae_finetune(
                run_name=run_name,
                pretext_backbone_path=cfg['pretext_source'],
                wandb_run=run,
            )
            dec_head, dec_hist = train_dec_finetune(
                run_name=run_name, ae_head=ae_head, k=cfg['k'], wandb_run=run,
            )
            # Backbone-only checkpoint = the encoder that build_index.py will load
            torch.save(dec_head.backbone.state_dict(), run_dir / 'backbone.pt')
            # KMeans/GMM/per-axis use the AE encoder's latent (matches contrastive script's eval shape)
            z = encode_all_latents(ae_head, DEVICE)
            eval_out = do_eval(z, dec_model=dec_head)

        else:
            raise ValueError(f'Unknown family: {family}')

        with open(evalj, 'w') as f:
            json.dump(eval_out, f, indent=2)
        print_eval_summary(run_name, eval_out)
        push_eval_to_wandb(run, eval_out, run_dir, run_name, family)
        if run is not None:
            print(f"[wandb] run URL: {run.url}")

    all_results[run_name] = eval_out
    elapsed = time.time() - _t_run
    print(f"[{run_name}] total wall-clock: {elapsed/60:.1f} min")


##############################################################################
# vae_z64  (family=vae, mode=offline)
##############################################################################


[15:33:55] training start: max 100 epochs, lr=0.001, weight_decay=1e-05, early_stop_patience=10, device=cuda
[15:34:04] epoch   1/100 | train=0.4333 val=0.2776 ↓ (best=inf) | 8.8s
[15:34:13] epoch   2/100 | train=0.3176 val=0.2727 ↓ (best=0.2776) | 9.6s
[15:34:22] epoch   3/100 | train=0.3201 val=0.2957 · (best=0.2727) | 8.2s
[15:34:31] epoch   4/100 | train=0.3417 val=0.3220 · (best=0.2727) | 9.7s
[15:34:41] epoch   5/100 | train=0.3668 val=0.3481 · (best=0.2727) | 9.6s
[15:34:49] epoch   6/100 | train=0.3899 val=0.3709 · (best=0.2727) | 8.1s
[15:34:58] epoch   7/100 | train=0.4112 val=0.3938 · (best=0.2727) | 9.5s
[15:35:07] epoch   8/100 | train=0.4307 val=0.4159 · (best=0.2727) | 8.6s
[15:35:16] epoch   9/100 | train=0.4497 val=0.4294 · (best=0.2727) | 8.8s
[15:35:25] epoch  10/100 | train=0.4659 val=0.4451 · (best=0.2727) | 9.4s
[15:35:34] epoch  11/100 | train=0.4637 val=0.4478 · (best=0.2727) | 8.2s
[15:35:43] epoch  12/100 | train=0.4623 val=0.4455 · (best=0.2727) | 9.5s
[15:35

wandb: WARNING URL not available in offline run


[vae_z64] KMeans  k=21    g=0.103  d=0.358  l=0.056
[vae_z64] GMM     k=21    g=0.077  d=0.190  l=0.098
[vae_z64] MVP baseline   g=0.332 (dec_z64_k21)
[wandb] run URL: None


axis_decade_ami_k12,▁
axis_decade_ari_k12,▁
axis_decade_nmi_k12,▁
axis_genre_ami_k21,▁
axis_genre_ari_k21,▁
axis_genre_nmi_k21,▁
axis_lang_ami_k11,▁
axis_lang_ari_k11,▁
axis_lang_nmi_k11,▁
best_val,█▁▁▁▁▁▁▁▁▁▁
+24,...


...

  [init] loaded pretext backbone from contrastive_tau0p5_drop0p3/
[16:01:42] training start: max 100 epochs, lr=0.001, weight_decay=1e-05, early_stop_patience=10, device=cuda
[16:01:51] epoch   1/100 | train=0.1982 val=0.0620 ↓ (best=inf) | 9.2s
[16:01:59] epoch   2/100 | train=0.0819 val=0.0457 ↓ (best=0.0620) | 8.3s
[16:02:08] epoch   3/100 | train=0.0663 val=0.0349 ↓ (best=0.0457) | 8.5s
[16:02:17] epoch   4/100 | train=0.0594 val=0.0319 ↓ (best=0.0349) | 9.6s
[16:02:25] epoch   5/100 | train=0.0541 val=0.0305 ↓ (best=0.0319) | 7.7s
[16:02:34] epoch   6/100 | train=0.0513 val=0.0283 ↓ (best=0.0305) | 9.2s
[16:02:43] epoch   7/100 | train=0.0486 val=0.0284 · (best=0.0283) | 9.0s
[16:02:52] epoch   8/100 | train=0.0473 val=0.0279 ↓ (best=0.0283) | 8.5s
[16:03:01] epoch   9/100 | train=0.0454 val=0.0279 · (best=0.0279) | 9.4s
[16:03:10] epoch  10/100 | train=0.0443 val=0.0249 ↓ (best=0.0279) | 8.2s
[16:03:19] epoch  11/100 | train=0.0432 val=0.0260 · (best=0.0249) | 9.3s
[16:03:28] epo

wandb: WARNING URL not available in offline run


[dec_z64_k21_from_contrastive_t0p5] KMeans  k=21    g=0.098  d=0.495  l=0.125
[dec_z64_k21_from_contrastive_t0p5] GMM     k=21    g=0.096  d=0.500  l=0.122
[dec_z64_k21_from_contrastive_t0p5] DEC argmax     g=0.098  d=0.487  l=0.125
[dec_z64_k21_from_contrastive_t0p5] MVP baseline   g=0.332 (dec_z64_k21)
[wandb] run URL: None


axis_decade_ami_k12,▁
axis_decade_ari_k12,▁
axis_decade_nmi_k12,▁
axis_genre_ami_k21,▁
axis_genre_ari_k21,▁
axis_genre_nmi_k21,▁
axis_lang_ami_k11,▁
axis_lang_ari_k11,▁
axis_lang_nmi_k11,▁
best_val,█▅▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+36,...


[dec_z64_k21_from_contrastive_t0p5] total wall-clock: 16.4 min


## Round 1 summary — pick winner by geo_NMI
Reads every eval.json on Drive (MVP carry-overs + Phase 1 contrastive + Round 1) and prints a single comparison table sorted by `geo_NMI`. The top row of the DEC-argmax sub-section is the Round 1 winner that feeds Round 2.

In [6]:
def _geo_nmi(d):
    if d is None:
        return None
    g = d.get('genre_nmi'); de = d.get('decade_nmi'); l = d.get('lang_nmi')
    if g is None or de is None or l is None:
        return None
    prod = max(g * de * l, 0.0)
    return (prod ** (1/3)) if prod > 0 else 0.0


# 1) Collect all eval.json on Drive (MVP carry-overs come from artifacts/eval/results.json)
rows = []

# MVP carry-overs from results.json
mvp_results = ARTIFACTS / 'eval' / 'results.json'
if mvp_results.exists():
    mvp = json.loads(mvp_results.read_text())
    for run_name, m in mvp.items():
        rows.append({
            'run':     run_name,
            'source':  'MVP',
            'gNMI':    m.get('genre_nmi'),
            'dNMI':    m.get('decade_nmi'),
            'lNMI':    m.get('lang_nmi'),
            'geo_NMI': _geo_nmi(m),
            'metric_source': 'KMeans/DEC (MVP)',
        })

# Per-run eval.json files (Phase 1 contrastive + Round 1)
for run_dir in sorted((ARTIFACTS / 'models').iterdir()):
    if not run_dir.is_dir():
        continue
    evalj = run_dir / 'eval.json'
    if not evalj.exists():
        continue
    e = json.loads(evalj.read_text())
    # Prefer DEC-argmax metric if present (Round 1 DEC family); else KMeans k=21.
    src_block = e.get('dec_argmax') or e.get('kmeans_k21') or {}
    src_label = 'DEC argmax' if 'dec_argmax' in e else 'KMeans k=21'
    if run_dir.name.startswith('contrastive_'):
        category = 'Phase 1 pretext'
    elif run_dir.name.startswith('dec_z64_k21_from_contrastive') or run_dir.name == 'vae_z64':
        category = 'Round 1'
    else:
        category = 'Other'
    rows.append({
        'run':     run_dir.name,
        'source':  category,
        'gNMI':    src_block.get('genre_nmi'),
        'dNMI':    src_block.get('decade_nmi'),
        'lNMI':    src_block.get('lang_nmi'),
        'geo_NMI': _geo_nmi(src_block),
        'metric_source': src_label,
    })

# 2) Print sorted by geo_NMI
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    df = df.sort_values('geo_NMI', ascending=False, na_position='last').reset_index(drop=True)
    cols = ['run', 'source', 'gNMI', 'dNMI', 'lNMI', 'geo_NMI', 'metric_source']
    print(df[cols].to_string(index=False, float_format=lambda x: f'{x:.3f}'))
except ImportError:
    for r in sorted(rows, key=lambda x: x['geo_NMI'] or 0, reverse=True):
        print(r)

# 3) Identify the Round 1 winner (highest geo_NMI among 'Round 1' rows)
round1_rows = [r for r in rows if r['source'] == 'Round 1' and r['geo_NMI'] is not None]
if round1_rows:
    winner = max(round1_rows, key=lambda x: x['geo_NMI'])
    print(f"\n=== Round 1 WINNER: {winner['run']}  geo_NMI={winner['geo_NMI']:.3f} ({winner['metric_source']}) ===")
    print('   → use this run as the Round 2 starting point.')
else:
    print('\n(no Round 1 results on Drive yet)')

if WANDB_MODE == 'offline':
    print(f"\nSync offline runs later with:  wandb sync {REPO_ROOT}/wandb/offline-run-*")

                              run          source  gNMI  dNMI  lNMI  geo_NMI    metric_source
                           ae_z64             MVP 0.328 0.341 0.264    0.309 KMeans/DEC (MVP)
       contrastive_tau0p5_drop0p3 Phase 1 pretext 0.216 0.218 0.374    0.260      KMeans k=21
       contrastive_tau0p1_drop0p3 Phase 1 pretext 0.216 0.286 0.174    0.221      KMeans k=21
                   vanilla_ae_z64             MVP 0.287 0.369 0.095    0.215 KMeans/DEC (MVP)
       contrastive_tau0p1_drop0p4 Phase 1 pretext 0.150 0.243 0.189    0.190      KMeans k=21
dec_z64_k21_from_contrastive_t0p5         Round 1 0.098 0.487 0.125    0.181       DEC argmax
                        ae_z64_w1             MVP 0.165 0.367 0.070    0.162 KMeans/DEC (MVP)
                          vae_z64         Round 1 0.103 0.358 0.056    0.127      KMeans k=21
dec_z64_k21_from_contrastive_t0p1         Round 1 0.120 0.641 0.016    0.107       DEC argmax

=== Round 1 WINNER: dec_z64_k21_from_contrastive_t0p5  geo_

## Next step — Round 2 (z-sweep on Round 1 winner)

Once the winner is locked, train two more runs at z=32 and z=128 in the winner's family:
- If winner = `dec_from_contrastive_tXX` → need fresh contrastive pretext at z=32/128 (the Phase 1 sweep was z=64 only), then AE→DEC fine-tune in the winner's family. Compute: ~50 min.
- If winner = `vae_z64` → just `vae_z32` and `vae_z128` cold-start. Compute: ~20 min.

Round 2 will be a separate notebook (`08_round2_zsweep.ipynb`) so that this notebook stays focused on the locked z=64 architecture comparison.

After Round 2 the overall winner becomes the backbone loaded by `scripts/build_index.py` (spec `2026-05-16-web-app-demo-design.md`) to encode all 329k films into the 64-dim latent index for the FastAPI demo.

## Diagnostic — investigate Round 1 underperformance

Round 1 fine-tune results came in BELOW the MVP `ae_z64` baseline (geo_NMI 0.309). Two things to check before deciding next steps:

1. Is the MVP `dec_z64_k21` (gNMI=0.332) actually in `results.json`? It's missing from the summary table above.
2. For each Round 1 DEC run: was the reported `dec_argmax` worse than `kmeans_k21` on the AE-finetuned latent? If yes, the DEC head is the failure point, not the contrastive pretext — and we could salvage the AE-stage encoder instead.

In [ ]:
# ── Diagnostic 1: MVP results.json — what runs are tracked? ───────────────────
mvp_path = ARTIFACTS / 'eval' / 'results.json'
if mvp_path.exists():
    mvp = json.loads(mvp_path.read_text())
    print(f"MVP results.json runs: {sorted(mvp.keys())}")
    print()
    for rn in ('dec_z64_k21', 'kmeans_raw_k21', 'pca_kmeans_k21'):
        m = mvp.get(rn)
        if m is None:
            print(f"  {rn:25s}  MISSING from results.json")
        else:
            print(f"  {rn:25s}  g={m.get('genre_nmi',float('nan')):.3f}  "
                  f"d={m.get('decade_nmi',float('nan')):.3f}  "
                  f"l={m.get('lang_nmi',float('nan')):.3f}")
else:
    print(f"NO results.json at {mvp_path}")

print()
print("="*72)
print()

# ── Diagnostic 2: AE-latent KMeans vs DEC argmax for each Round 1 DEC run ────
for rn in ('dec_z64_k21_from_contrastive_t0p1', 'dec_z64_k21_from_contrastive_t0p5'):
    p = ARTIFACTS / 'models' / rn / 'eval.json'
    if not p.exists():
        print(f"{rn}  -- eval.json missing")
        continue
    e = json.loads(p.read_text())
    km = e.get('kmeans_k21', {})
    gm = e.get('gmm_k21', {})
    pa = e.get('per_axis_k_kmeans', {})
    de = e.get('dec_argmax', {})
    print(f"{rn}")
    print(f"  KMeans on AE latent  g={km.get('genre_nmi',0):.3f}  "
          f"d={km.get('decade_nmi',0):.3f}  l={km.get('lang_nmi',0):.3f}")
    print(f"  GMM on AE latent     g={gm.get('genre_nmi',0):.3f}  "
          f"d={gm.get('decade_nmi',0):.3f}  l={gm.get('lang_nmi',0):.3f}")
    print(f"  per-axis-k KMeans    g={pa.get('genre_nmi_k21',0):.3f}  "
          f"d={pa.get('decade_nmi_k12',0):.3f}  l={pa.get('lang_nmi_k11',0):.3f}")
    print(f"  DEC argmax           g={de.get('genre_nmi',0):.3f}  "
          f"d={de.get('decade_nmi',0):.3f}  l={de.get('lang_nmi',0):.3f}")
    # Geo NMI on each method
    for tag, src in (('KMeans', km), ('DEC', de)):
        g, d, l = src.get('genre_nmi'), src.get('decade_nmi'), src.get('lang_nmi')
        if None not in (g, d, l):
            prod = max(g*d*l, 0.0)
            geo = prod**(1/3) if prod > 0 else 0.0
            print(f"  geo_NMI [{tag:6s}]      {geo:.3f}")
    print()